In [ ]:
def construct_laplacian_matrix():
    A = np.zeros([Nx*Ny, Nx*Ny])

    Bx = D * dt / dx**2
    By = D * dt / dy**2
    A_const = -2 * (Bx + By) + 1
    
    for j in range(1, Ny-1):
        for i in range(1, Nx-1):
            n = j*Nx + i
            A[n][n] = A_const #U(i,j)
            A[n][n-Nx] = By #U(i,j-1)
            A[n][n-1] = Bx #U(i-1,j)
            A[n][n+1] = Bx #U(i+1,j)
            A[n][n+Nx] = By #U(i,j+1)

    return A


In [ ]:

Lx = 0.6
Ly = 0.6

Nx = 5
Ny = 4
D = 2e-5 # constant for air. units m^2 / s
u0 = 20
Tmax = 4000

dx = Lx/(Nx-1)
dy = Ly/(Ny-1)
    rx = D * dt / dx**2
    ry = D * dt / dy**2

dt = 1 / (2*D* (1/dx**2 + 1/ dy**2)) #CFL condition

u = np.zeros([Ny, Nx]) + u0
max_iters = int(round(Tmax/dt))

## boundary / initial conditions

F_bottom = lambda x: 100
F_top = lambda x: 100
F_left = lambda y: 100
F_right = lambda y: 0
X = np.linspace(0, Lx, Nx)
Y = np.linspace(0, Ly, Ny)

u[0, :] = F_bottom(X)
u[-1, :] = F_top(X)
u[:, 0] = F_left(Y)
u[:, -1] = F_right(Y)
#mpx, mpy = int(Nx/2), int(Ny/2) #midpoint
#u[mpy-5:mpy+5, mpx-5:mpx+5] = 100


def construct_laplacian_matrix(): # implicit euler
    A = np.zeros([Nx*Ny, Nx*Ny])

    rx = D * dt / dx**2
    ry = D * dt / dy**2
    A_const = 1 + 2 * (rx + ry)
    
    for j in range(1, Ny-1):
        for i in range(1, Nx-1):
            n = j*Nx + i
            A[n][n] = A_const #U(i,j)
            A[n][n-Nx] = -ry #U(i,j-1)
            A[n][n-1] = -rx #U(i-1,j)
            A[n][n+1] = -rx #U(i+1,j)
            A[n][n+Nx] = -ry #U(i,j+1)

    return A

A_mat = construct_laplacian_matrix()
A_inv = np.linalg.inv(A_mat)


LinAlgError: Singular matrix

In [19]:
D * dt / dx**2

0.32000000000000006

In [2]:
import numpy as np
import matplotlib.pyplot as plt

Lx = 50
Ly = 40

Nx = 5
Ny = 4
dx = Lx/(Nx-1)
dy = Ly/(Ny-1)

D = 1
dt = min(dx**2 / (4*D), dy**2 / (4*D)) #CFL condition

u = np.zeros([Ny, Nx])

# using global variables

def construct_matrix():
    A = np.zeros([Nx*Ny, Nx*Ny])

    A_const = 2 * D * dt * (-1/dx**2 - 1/dy**2) + 1
    Bx = D * dt / dx**2
    By = D * dt / dy**2
    
    for j in range(0, Ny):
        for i in range(0, Nx):
            n = j*Nx + i

            if j == 0 and i == 0: #bottom-left corner
                A[n][n] = 1

            elif j == 0 and i == Nx-1: #bottom-right corner
                A[n][n] = 1

            elif j == Ny-1 and i == 0: #top-left corner
                A[n][n] = 1

            elif j == Ny-1 and i == Nx-1: #top-right corner
                A[n][n] = 1

            elif j == 0: #bottom
                A[n][n] = 1

            elif j == Ny-1: #top
                A[n][n] = 1

            elif i == 0: #left
                A[n][n] = 1

            elif i == Nx-1: #right
                A[n][n] = 1

            else: #interior points
                A[n][n] = A_const #U(i,j)
                A[n][n-Nx] = By #U(i,j-1)
                A[n][n-1] = Bx #U(i-1,j)
                A[n][n+1] = Bx #U(i+1,j)
                A[n][n+Nx] = By #U(i,j+1)
        
    return A

def construct_matrix_homogenous_neumann():
    A = np.zeros([Nx*Ny, Nx*Ny])

    A_const = 2 * D * dt * (-1/dx**2 - 1/dy**2) + 1
    Bx = D * dt / dx**2
    By = D * dt / dy**2
    
    for j in range(0, Ny):
        for i in range(0, Nx):
            n = j*Nx + i

            if j == 0 and i == 0: #bottom-left corner
                A[n][n] = A_const
                A[n][n+1] = 2*Bx
                A[n][n+Nx] = 2*By

            elif j == 0 and i == Nx-1: #bottom-right corner
                A[n][n] = A_const
                A[n][n-1] = 2*Bx
                A[n][n+Nx] = 2*By

            elif j == Ny-1 and i == 0: #top-left corner
                A[n][n] = A_const
                A[n][n+1] = 2*Bx
                A[n][n-Nx] = 2*By

            elif j == Ny-1 and i == Nx-1: #top-right corner
                A[n][n] = A_const
                A[n][n-1] = 2*Bx
                A[n][n-Nx] = 2*By

            elif j == 0: #bottom
                A[n][n] = A_const
                A[n][n-1] = Bx
                A[n][n+1] = Bx
                A[n][n+Nx] = 2*By

            elif j == Ny-1: #top
                A[n][n] = A_const
                A[n][n-Nx] = 2*By
                A[n][n-1] = Bx
                A[n][n+1] = Bx

            elif i == 0: #left
                A[n][n] = A_const
                A[n][n-Nx] = By
                A[n][n+1] = 2*Bx
                A[n][n+Nx] = By

            elif i == Nx-1: #right
                A[n][n] = A_const
                A[n][n-Nx] = By
                A[n][n-1] = 2*Bx
                A[n][n+Nx] = By

            else: #interior points
                A[n][n] = A_const #U(i,j)
                A[n][n-Nx] = By #U(i,j-1)
                A[n][n-1] = Bx #U(i-1,j)
                A[n][n+1] = Bx #U(i+1,j)
                A[n][n+Nx] = By #U(i,j+1)

    return A

arr = construct_matrix_homogenous_neumann()

In [8]:
np.arange(1,4)

array([1, 2, 3])

In [10]:
def boundary_vector_dirichlet():
    b = np.zeros(Nx*Ny)
    test = np.zeros([Ny,Nx])
    n_idx = lambda i, j: j*Nx + i

    # bottom
    i = np.arange(0,Nx)
    j = 0
    n = n_idx(i, j)
    test[j,i] = 1
    b[n] = u[j,i]

    # top
    i = np.arange(0,Nx)
    j = Ny-1
    n = n_idx(i, j)
    test[j,i] = 1
    b[n] = u[j,i]

    # left
    i = 0
    j = np.arange(0,Ny)
    n = n_idx(i, j)
    test[j,i] = 1
    b[n] = u[j,i]

    # right
    i = Nx-1
    j = np.arange(0,Ny)
    n = n_idx(i, j)
    test[j,i] = 1
    b[n] = u[j,i]

    return b, test
    
boundary_vector_dirichlet()

(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0.]),
 array([[1., 1., 1., 1., 1.],
        [1., 0., 0., 0., 1.],
        [1., 0., 0., 0., 1.],
        [1., 1., 1., 1., 1.]]))

In [7]:
def boundary_vector_dirichlet():
    b = np.zeros(Nx*Ny)
    test = np.zeros([Ny,Nx])
    n_idx = lambda i, j: j*Nx + i

    for j in range(0, Ny):
        if j == 0 or j == Ny-1:
            for i in range(0, Nx):
                n = n_idx(i, j)
                b[n] = 1
                test[j,i] = 1
        
        n = n_idx(0,j)
        b[n] = 1
        test[j,0] = 1
        n = n_idx(Nx-1,j)
        b[n] = 1
        test[j,Nx-1] = 1
    
    print(test)
    return b
boundary_vector_dirichlet()

[[1. 1. 1. 1. 1.]
 [1. 0. 0. 0. 1.]
 [1. 0. 0. 0. 1.]
 [1. 1. 1. 1. 1.]]


array([1., 1., 1., 1., 1., 1., 0., 0., 0., 1., 1., 0., 0., 0., 1., 1., 1.,
       1., 1., 1.])

In [112]:
u0 = np.random.rand(Nx*Ny)

In [118]:
u1 = u0.copy()

for _ in range(100):
    u1 = arr @ u1

print(u1.sum() - u0.sum())

-0.06336400948262089


In [38]:
# length in both directions in m
Lx = 0.7
Ly = 0.6

Nx = 50
Ny = 40
D = 2e-5 # constant for air. units m^2 / s
u0 = 0

dx = Lx/(Nx-1)
dy = Ly/(Ny-1)
dt = min(dx**2 / (4*D), dy**2 / (4*D)) #CFL condition

u = np.zeros([Ny, Nx]) + u0

Tmax = 1000
max_iters = int(round(Tmax/dt))
max_iters, dt

(392, 2.551020408163265)

In [229]:
def construct_laplacian_matrix_interior():
    N = (Nx-2)*(Ny-2)
    A = np.zeros([N, N])

    rx = D * dt / dx**2
    ry = D * dt / dy**2
    A_const = 1 + 2 * (rx + ry)
    
    for j in range(0, Ny-2):
        for i in range(0, Nx-2):
            n = 1*(Nx-2) + i
            A[n][n] = A_const       #U(i,j)
            A[n][n-(Nx-2)] = -ry    #U(i,j-1)
            A[n][n-1] = -rx         #U(i-1,j)
            A[n][n+1] = -rx         #U(i+1,j)
            A[n][n+(Nx-2)] = -ry    #U(i,j+1)

    return A

arr = construct_laplacian_matrix_interior()
arr

array([[ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [-0.32,  0.  , -0.32,  2.28, -0.32,  0.  , -0.32,  0.  ,  0.  ],
       [ 0.  , -0.32,  0.  , -0.32,  2.28, -0.32,  0.  , -0.32,  0.  ],
       [ 0.  ,  0.  , -0.32,  0.  , -0.32,  2.28, -0.32,  0.  , -0.32],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ]])

In [326]:
def vec_to_grid(v, N_x, N_y):
    a = np.zeros((N_x, N_y))
    for j in range(N_y):
        for i in range(N_x):
            n = j*(Nx-2) + i
            a[j,i] = v[n]
    return a

In [327]:
u = np.zeros((Ny, Nx))
u[3, 2] = 40
u

array([[ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0., 40.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.]])

In [401]:
rx = D * dt / dx**2
ry = D * dt / dy**2

boundary_bottom = 20
boundary_top = 20
boundary_left = 20
boundary_right = 20

u[0, :] = boundary_bottom
u[-1, :] = boundary_top
u[:, 0] = boundary_left
u[:, -1] = boundary_right

interior = u[1:Ny-1, 1:Nx-1]
interior[0, :] = ry * boundary_bottom # bottom interior
interior[-1, :] = ry * boundary_top # top interior
interior[:, 0] = rx * boundary_left # left interior
interior[:, -1] = rx * boundary_right # right interiod

interior_vec = interior.flatten() # values of the interior points

new_interior = A_inv @ interior_vec
u[1:Ny-1, 1:Nx-1] = vec_to_grid(new_interior, Nx-2, Ny-2)

u

array([[20. , 20. , 20. , 20. , 20. ],
       [20. ,  6.4,  6.4,  6.4, 20. ],
       [20. ,  6.4,  6.4,  6.4, 20. ],
       [20. ,  6.4,  6.4,  6.4, 20. ],
       [20. , 20. , 20. , 20. , 20. ]])

In [1098]:
Nx, Ny = 5,5
dx = Lx/(Nx-1)
dy = Ly/(Ny-1)

u = np.zeros((Ny, Nx))
u[2:4, 2:4] = 40
u1 = u.copy()
u

array([[ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0., 40., 40.,  0.],
       [ 0.,  0., 40., 40.,  0.],
       [ 0.,  0.,  0.,  0.,  0.]])

In [1110]:
## implicit euler - neumann boundary conditions

def construct_matrix(): # implicit euler
    N = (Nx-2)*(Ny-2)
    A = np.zeros([N, N])

    rx = D * dt / dx**2
    ry = D * dt / dy**2
    
    for j in range(Ny-2):
        for i in range(Nx-2):
            n = j*(Nx-2) + i
            
            if 0 < i < Nx-3 and 0 < j < Ny-3:
                A[n][n] = 1 + 2 * (rx + ry)

            elif i == 0 and j == 0:
                A[n][n] = 1 + 3*rx + 3*ry

            elif i == 0 and j == Ny-3:
                A[n][n] = 1 + 3*rx + 3*ry

            elif i == Nx-3 and j == 0:
                A[n][n] = 1 + 3*rx + 3*ry

            elif i == Nx-3 and j == Ny-3:
                A[n][n] = 1 + 3*rx + 3*ry
            
            elif j == 0 or j == Ny-3:
                A[n][n] = 1 + 2*rx + 3*ry

            elif i == 0 or i == Nx-3:
                A[n][n] = 1 + 3*rx + 2*ry

            if j > 0: #bottom
                A[n][n-(Nx-2)] = -ry

            if j < Ny-3: #top
                A[n][n+(Nx-2)] = -ry

            if i > 0: #left
                A[n][n-1] = -rx

            if i < Nx-3: #right
                A[n][n+1] = -rx

    return A

a_inv = np.linalg.inv(construct_matrix())

g = lambda x, y, t: 0
t = 0

rx = D * dt / dx**2
ry = D * dt / dy**2

ghostcells_bottom = u[1, 1:-1] - 2*dy*g(dx*np.arange(1, Nx-1), 0, t+dt)
ghostcells_top = u[-2, 1:-1] + 2*dy*g(dx*np.arange(1, Nx-1), Ly, t+dt)
ghostcells_left = u[1:-1, 1] - 2*dx*g(0, dy*np.arange(1, Ny-1), t+dt)
ghostcells_right = u[1:-1, -2] + 2*dx*g(Lx, dy*np.arange(1, Ny-1), t+dt)

interior = u.copy()[1:-1, 1:-1]
interior[0, :] += ry * ghostcells_bottom # bottom interior row
interior[-1, :] += ry * ghostcells_top # top interior row
interior[:, 0] += rx * ghostcells_left # left interior col
interior[:, -1] += rx * ghostcells_right # right interior col

interior_vec = interior.flatten() # values of the interior points

new_interior = a_inv @ interior_vec
u[1:Ny-1, 1:Nx-1] = new_interior.reshape(Nx-2, Ny-2)

u

array([[0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.64594325, 0.90543104, 0.75291217, 0.        ],
       [0.        , 0.90543104, 1.25729029, 1.0542481 , 0.        ],
       [0.        , 0.75291217, 1.0542481 , 0.89928977, 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ]])

In [1069]:
((u1.sum()-u.sum())/u1.sum())*100

np.float64(7.476635513972455)

In [643]:
u2

array([[ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0., 40., 40.,  0.],
       [ 0.,  0., 40., 40.,  0.],
       [ 0.,  0.,  0.,  0.,  0.]])

In [ ]:
b[0::(Nx-2)] += rx * (u[1,1:-1] - 2*dx*g(0, dy*np.arange(1,Ny-1), t+dt))  

# Right i=Nx-1  
b[(Nx-3)::(Nx-2)] += rx * (u[-2,1:-1] + 2*dx*g(Lx, dy*np.arange(1,Ny-1), t+dt))

# Bottom j=0  
b[0:Nx-2] += ry * (u[1:-1,1] - 2*dy*g(dx*np.arange(1,Nx-1), 0, t+dt))

# Top j=Ny-1  
b[(Ny-3)*(Nx-2)::(Nx-2)] += ry * (u[1:-1,-2] + 2*dy*g(dx*np.arange(1,Nx-1), Ly, t+dt))

---

In [1328]:

g = lambda x, y, t: 0



def neumann_matrix(**kwargs): # using global variables
    A = np.zeros([Nx*Ny, Nx*Ny])

    rx = D * dt / dx**2
    ry = D * dt / dy**2
    A_const = 1 + 2 * (rx + ry)
    
    for j in range(0, Ny):
        for i in range(0, Nx):
            n = j*Nx + i

            if j == 0 and i == 0: #bottom-left corner
                A[n][n] = 1

            elif j == 0 and i == Nx-1: #bottom-right corner
                A[n][n] = 1

            elif j == Ny-1 and i == 0: #top-left corner
                A[n][n] = 1

            elif j == Ny-1 and i == Nx-1: #top-right corner
                A[n][n] = 1

            elif j == 0: #bottom
                A[n][n] = 1

            elif j == Ny-1: #top
                A[n][n] = 1

            elif i == 0: #left
                A[n][n] = 1

            elif i == Nx-1: #right
                A[n][n] = 1

            else: #interior points
                A[n][n] = A_const #U(i,j)
                A[n][n-Nx] = -ry #U(i,j-1)
                A[n][n-1] = -rx #U(i-1,j)
                A[n][n+1] = -rx #U(i+1,j)
                A[n][n+Nx] = -ry #U(i,j+1)

            ## wraparound?
    return A

#np.linalg.inv(neumann_matrix())
np.linalg.inv(neumann_matrix()) @ T.flatten()

array([5., 5., 5., 5., 5., 5., 5., 5., 5., 5., 5., 5., 5., 5., 5., 5., 5.,
       5., 5., 5., 5., 5., 5., 5., 5.])

In [1324]:
T2 = T.copy()
T2[2:4, 2] = 10
T2

array([[ 5.,  5.,  5.,  5.,  5.],
       [ 5.,  5.,  5.,  5.,  5.],
       [ 5.,  5., 10.,  5.,  5.],
       [ 5.,  5., 10.,  5.,  5.],
       [ 5.,  5.,  5.,  5.,  5.]])

In [1322]:
T2 = (np.linalg.inv(neumann_matrix()) @ T2.flatten()) + neumann_boundaries(T2.reshape(Ny, Nx))
T2

array([   5.        ,  574.27742436,  287.1910361 ,  574.27742436,
          5.        ,  574.42780531,   84.54426258,   57.56111889,
         84.54426258,  574.42780531,  288.16446791,   57.74529855,
         36.62554756,   57.74529855,  288.16446791,  565.72431118,
         85.86573745,   36.25396986,   85.86573745,  565.72431118,
          5.        ,  728.14431549, -172.5966707 ,  728.14431549,
          5.        ])

In [1297]:

def neumann_boundaries(u):
    b = np.zeros(Nx*Ny)
    rx = D * dt / dx**2
    ry = D * dt / dy**2
    
    # bottom row

    j = 0
    i = np.arange(1, Nx-1)
    n = j*Nx + i
    ghostcells = u[1, 1:-1] - 2*dy*g(dx*np.arange(1, Nx-1), 0, t+dt)
    b[n] = (1 + 2 * (rx + ry)) * u[j,i] - ry * ghostcells - rx * u[j,i-1] - rx * u[j,i+1] - ry * u[j+1,i]

    # top row

    j = Ny-1
    i = np.arange(1, Nx-1)
    n = j*Nx + i
    ghostcells = u[-2, 1:-1] + 2*dy*g(dx*np.arange(1, Nx-1), Ly, t+dt)
    b[n] = (1 + 2 * (rx + ry)) * u[j,i] - ry * u[j-1, i] - rx * u[j,i-1] - rx * u[j,i+1] - ry * ghostcells

    # left row

    j = np.arange(1, Ny-1)
    i = 0
    n = j*Nx + i
    ghostcells = u[1:-1, 1] - 2*dx*g(0, dy*np.arange(1, Ny-1), t+dt)
    b[n] = (1 + 2 * (rx + ry)) * u[j,i] - ry * u[j-1, i] - rx * ghostcells - rx * u[j,i+1] - ry * u[j+1,i]

    # right row

    j = np.arange(1, Ny-1)
    i = Nx-1
    n = j*Nx + i
    ghostcells = u[1:-1, -2] + 2*dx*g(Lx, dy*np.arange(1, Ny-1), t+dt)
    b[n] = (1 + 2 * (rx + ry)) * u[j,i] - ry * u[j-1, i] - rx * u[j,i-1] - rx * ghostcells - ry * u[j+1,i]

    #1 + 3*rx + 3*ry
    # bottom left corner
    # bottom right corner
    # top left corner
    # top right corner

    return b

neumann_boundaries(T)

array([0., 5., 5., 5., 0., 5., 0., 0., 0., 5., 5., 0., 0., 0., 5., 5., 0.,
       0., 0., 5., 0., 5., 5., 5., 0.])

In [ ]:

a_inv = np.linalg.inv(construct_matrix())

g = lambda x, y, t: 0
t = 0

rx = D * dt / dx**2
ry = D * dt / dy**2



In [1332]:
neumann_matrix(Nx=3, Ny=3)

array([[ 1.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  1.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  1.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  ,  1.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  1.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
